<a href="https://colab.research.google.com/github/bharath-hue/TASK_TNS_AIML/blob/main/Rainfall_Dataset_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()

file_name = list(uploaded.keys())[0]
print("Uploaded file:", file_name)

Saving rainfall.csv to rainfall (1).csv
Uploaded file: rainfall (1).csv


Dataset Link: https://www.kaggle.com/datasets/sujithmandala/simple-rainfall-classification-dataset?utm_source=chatgpt.com

In [3]:
df = pd.read_csv(file_name)
df.head()

,date,rainfall,temperature,humidity,wind_speed,weather_condition
0,2022-01-01,12.5,15.2,78.0,8.5,Rainy
1,2022-01-02,8.2,17.8,65.0,5.2,Rainy
2,2022-01-03,0.0,20.1,52.0,3.1,Sunny
3,2022-01-04,3.7,18.6,71.0,6.7,Rainy
4,2022-01-05,21.1,14.8,82.0,9.3,Rainy


In [4]:
df.head()

df.info()

df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   date               54 non-null     object 
 1   rainfall           53 non-null     float64
 2   temperature        53 non-null     float64
 3   humidity           53 non-null     float64
 4   wind_speed         53 non-null     float64
 5   weather_condition  53 non-null     object 
dtypes: float64(4), object(2)
memory usage: 2.7+ KB


,0
date,0
rainfall,1
temperature,1
humidity,1
wind_speed,1
weather_condition,1


In [7]:
df["date"] = df["date"].astype(str).str.strip()
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   date               53 non-null     datetime64[ns]
 1   rainfall           53 non-null     float64       
 2   temperature        53 non-null     float64       
 3   humidity           53 non-null     float64       
 4   wind_speed         53 non-null     float64       
 5   weather_condition  53 non-null     object        
dtypes: datetime64[ns](1), float64(4), object(1)
memory usage: 2.7+ KB


In [18]:
num_cols = ["rainfall", "temperature", "humidity", "wind_speed"]

df[num_cols] = df[num_cols].fillna(df[num_cols].mean())

df["weather_condition"] = df["weather_condition"].fillna(df["weather_condition"].mode()[0])

print(df.isnull().sum())

date                 0
rainfall             0
temperature          0
humidity             0
wind_speed           0
weather_condition    0
dtype: int64


In [21]:
df["weather_condition"] = df["weather_condition"].str.lower()

df["weather_condition"].unique()
df

,date,rainfall,temperature,humidity,wind_speed,weather_condition
0,2022-01-01,12.5,15.2,78.0,8.5,rainy
1,2022-01-02,8.2,17.8,65.0,5.2,rainy
2,2022-01-03,0.0,20.1,52.0,3.1,sunny
3,2022-01-04,3.7,18.6,71.0,6.7,rainy
4,2022-01-05,21.1,14.8,82.0,9.3,rainy
5,2022-01-06,15.3,16.5,75.0,7.8,rainy
6,2022-01-07,6.8,19.2,61.0,4.5,rainy
7,2022-01-08,0.0,21.7,48.0,2.9,sunny
8,2022-01-09,11.2,17.3,73.0,6.1,rainy
9,2022-01-10,18.6,15.8,79.0,8.9,rainy


In [22]:
df = df.dropna(subset=['date'])

In [25]:
print("Duplicate rows:", df.duplicated().sum())
df.drop_duplicates(inplace=True)

Duplicate rows: 0


In [30]:
num_cols = ["rainfall", "temperature", "humidity", "wind_speed"]

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df = df[(df[col] >= lower) & (df[col] <= upper)]

print("Shape after removing outliers:", df.shape)

Shape after removing outliers: (53, 9)


In [31]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day

df.head()

,date,rainfall,temperature,humidity,wind_speed,weather_condition,year,month,day
0,2022-01-01,12.5,15.2,78.0,8.5,rainy,2022,1,1
1,2022-01-02,8.2,17.8,65.0,5.2,rainy,2022,1,2
2,2022-01-03,0.0,20.1,52.0,3.1,sunny,2022,1,3
3,2022-01-04,3.7,18.6,71.0,6.7,rainy,2022,1,4
4,2022-01-05,21.1,14.8,82.0,9.3,rainy,2022,1,5


In [32]:
df = pd.get_dummies(df, columns=["weather_condition"], drop_first=True)

df.head()

,date,rainfall,temperature,humidity,wind_speed,year,month,day,weather_condition_sunny
0,2022-01-01,12.5,15.2,78.0,8.5,2022,1,1,False
1,2022-01-02,8.2,17.8,65.0,5.2,2022,1,2,False
2,2022-01-03,0.0,20.1,52.0,3.1,2022,1,3,True
3,2022-01-04,3.7,18.6,71.0,6.7,2022,1,4,False
4,2022-01-05,21.1,14.8,82.0,9.3,2022,1,5,False


In [33]:
print(df.info())
print(df.isnull().sum())
print(df.head())

<class 'pandas.core.frame.DataFrame'>
Index: 53 entries, 0 to 52
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   date                     53 non-null     datetime64[ns]
 1   rainfall                 53 non-null     float64       
 2   temperature              53 non-null     float64       
 3   humidity                 53 non-null     float64       
 4   wind_speed               53 non-null     float64       
 5   year                     53 non-null     int32         
 6   month                    53 non-null     int32         
 7   day                      53 non-null     int32         
 8   weather_condition_sunny  53 non-null     bool          
dtypes: bool(1), datetime64[ns](1), float64(4), int32(3)
memory usage: 3.2 KB
None
date                       0
rainfall                   0
temperature                0
humidity                   0
wind_speed                 0
year

In [35]:
df.to_csv("rainfall_cleaned.csv", index=False)
files.download("rainfall_cleaned.csv")
print("Cleaned dataset saved successfully!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Cleaned dataset saved successfully!
